# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krihna7/flyrank-ML-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## Ranked actions + reason codes

The action queue ranks content items by the Random Forest probability of the defined refresh-risk proxy. The model score is used to prioritize limited human review capacity, not to automatically decide that a page must be refreshed.

I use four practical review actions:

- **Refresh review** — highest-priority items with high model probability, especially when the page is also stale and visible.
- **Improve review** — high-priority items where the model indicates refresh risk but the staleness or visibility signal is less strong.
- **Monitor** — moderate-risk items that should be watched before a refresh decision is made.
- **Protect / monitor** — lower-risk items where the model provides less evidence for immediate refresh review.

Reason codes combine the model score with observed visibility and freshness signals. They are explanations for prioritization, not causal explanations of why a page changed.

The queue should always be reviewed by a human before any editorial action.

In [23]:
# ML-10 Section 1 — Ranked actions + reason codes

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier

# --------------------------------------------------
# 1. Paths and data
# --------------------------------------------------

REPO = Path("/content/flyrank-ml-internship")
DATA_PATH = REPO / "data/raw/content_refresh_anonymized.csv"
OUTPUT_DIR = REPO / "work/outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

# --------------------------------------------------
# 2. Define target and final validated features
# --------------------------------------------------

df["target_refresh"] = (
    pd.to_numeric(df["trend_pct"], errors="coerce") < -10
).astype(int)

features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "pageviews_90d",
    "users_90d",
    "engaged_sessions_90d",
    "engagement_rate",
    "days_since_last_update"
]

features = [c for c in features if c in df.columns]

client_candidates = ["client_hash_id", "client_id"]
client_column = next(
    (c for c in client_candidates if c in df.columns),
    None
)

assert client_column is not None, "Client grouping column not found."

# --------------------------------------------------
# 3. Prepare modeling data
# --------------------------------------------------

model_df = df.copy()

for col in features:
    model_df[col] = pd.to_numeric(
        model_df[col],
        errors="coerce"
    )

model_df = model_df.dropna(
    subset=[client_column, "content_id"]
).copy()

# --------------------------------------------------
# 4. Grouped validation split
#    Same design as ML-09
# --------------------------------------------------

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        groups=model_df[client_column]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

# --------------------------------------------------
# 5. Training-only imputation
# --------------------------------------------------

for col in features:
    median_value = train_df[col].median()

    train_df[col] = train_df[col].fillna(median_value)
    test_df[col] = test_df[col].fillna(median_value)

# --------------------------------------------------
# 6. Train validated Random Forest
# --------------------------------------------------

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf.fit(
    train_df[features],
    train_df["target_refresh"]
)

test_df["model_probability"] = rf.predict_proba(
    test_df[features]
)[:, 1]

# --------------------------------------------------
# 7. Train final queue model on all available data
#
# This model is used only to generate the action queue.
# It is not used to claim validation performance.
# --------------------------------------------------

queue_df = model_df.copy()

for col in features:
    queue_df[col] = queue_df[col].fillna(
        model_df[col].median()
    )

final_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

final_rf.fit(
    queue_df[features],
    queue_df["target_refresh"]
)

queue_df["model_probability"] = final_rf.predict_proba(
    queue_df[features]
)[:, 1]

# --------------------------------------------------
# 8. Supporting scores
# --------------------------------------------------

queue_df["visibility_score"] = (
    queue_df["impressions_90d"]
    .rank(method="average", pct=True)
)

queue_df["staleness_score"] = (
    queue_df["days_since_last_update"]
    .rank(method="average", pct=True)
)

# --------------------------------------------------
# 9. Reason codes
# --------------------------------------------------

queue_df["reason_code"] = np.select(
    [
        (queue_df["model_probability"] >= 0.75)
        & (queue_df["staleness_score"] >= 0.75)
        & (queue_df["visibility_score"] >= 0.75),

        (queue_df["model_probability"] >= 0.75)
        & (queue_df["staleness_score"] >= 0.75),

        (queue_df["model_probability"] >= 0.75)
        & (queue_df["visibility_score"] >= 0.75),

        queue_df["model_probability"] >= 0.75,

        queue_df["model_probability"] >= 0.50,

        queue_df["model_probability"] >= 0.25
    ],
    [
        "high_risk_stale_high_visibility",
        "high_risk_stale",
        "high_risk_high_visibility",
        "high_model_refresh_risk",
        "moderate_model_refresh_risk",
        "lower_model_refresh_risk"
    ],
    default="low_model_refresh_risk"
)

# --------------------------------------------------
# 10. Action mapping
# --------------------------------------------------

queue_df["action"] = np.select(
    [
        queue_df["reason_code"]
        == "high_risk_stale_high_visibility",

        queue_df["model_probability"] >= 0.75,

        queue_df["model_probability"] >= 0.50,

        queue_df["model_probability"] >= 0.25
    ],
    [
        "refresh_review",
        "improve_review",
        "monitor",
        "protect_monitor"
    ],
    default="protect_monitor"
)

# --------------------------------------------------
# 11. Rank the queue
# --------------------------------------------------

queue_df = queue_df.sort_values(
    [
        "model_probability",
        "staleness_score",
        "visibility_score"
    ],
    ascending=[False, False, False]
).reset_index(drop=True)

queue_df["rank"] = np.arange(1, len(queue_df) + 1)

# --------------------------------------------------
# 12. Select paper-ready queue columns
# --------------------------------------------------

action_queue = queue_df[
    [
        "rank",
        "content_id",
        client_column,
        "model_probability",
        "action",
        "reason_code",
        "days_since_last_update",
        "impressions_90d",
        "visibility_score",
        "staleness_score"
    ]
].copy()

action_queue["model_probability"] = (
    action_queue["model_probability"].round(4)
)

action_queue["visibility_score"] = (
    action_queue["visibility_score"].round(4)
)

action_queue["staleness_score"] = (
    action_queue["staleness_score"].round(4)
)

print("\nACTION QUEUE")
print("=" * 60)

print("Rows:", len(action_queue))

print("\nAction counts:")
display(
    action_queue["action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="count")
)

print("\nTop 20 ranked actions:")
display(action_queue.head(20))

Dataset shape: (30000, 44)

ACTION QUEUE
Rows: 30000

Action counts:


,action,count
0,monitor,18995
1,protect_monitor,10123
2,improve_review,609
3,refresh_review,273



Top 20 ranked actions:


,rank,content_id,client_id,model_probability,action,reason_code,days_since_last_update,impressions_90d,visibility_score,staleness_score
0,1,content_6ff59596237d,client_6208ef0f77,0.8322,improve_review,high_risk_stale,104,506,0.4442,0.8432
1,2,content_45c6b7efaa2b,client_6208ef0f77,0.8322,improve_review,high_risk_stale,104,563,0.4600,0.8432
2,3,content_6da8cc871a23,client_6208ef0f77,0.8316,improve_review,high_risk_stale,104,668,0.4850,0.8432
3,4,content_4b254381379e,client_6208ef0f77,0.8311,improve_review,high_risk_stale,104,903,0.5344,0.8432
4,5,content_9b6df29f7889,client_3fdba35f04,0.8291,improve_review,high_risk_stale,104,1622,0.6256,0.8432
5,6,content_98843f898048,client_6208ef0f77,0.8269,improve_review,high_risk_stale,104,712,0.4954,0.8432
6,7,content_51a87e64469f,client_6208ef0f77,0.8252,improve_review,high_risk_stale,104,931,0.5387,0.8432
7,8,content_0ba896169af5,client_4e07408562,0.8251,refresh_review,high_risk_stale_high_visibility,104,4998,0.7949,0.8432
8,9,content_11ba829169ac,client_19581e27de,0.8249,refresh_review,high_risk_stale_high_visibility,104,6124,0.8217,0.8432
9,10,content_1052cb379f1d,client_6208ef0f77,0.8242,improve_review,high_risk_stale,104,404,0.4137,0.8432


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## Intended use and limits

This playbook is intended for SEO or content teams that need to prioritize a limited number of pages for human review.

The model provides a ranked decision-support queue based on observed performance, engagement, ranking, and freshness signals. The highest-ranked items are candidates for review first.

The measured 80.0% Precision@50 comes from the client-grouped validation design used in ML-09 and should not be interpreted as a guarantee of future performance. The Week-4 baseline recorded 52.0% Precision@50 on the same held-out test set, giving a measured difference of 28.0 percentage points.

The target is a rule-derived proxy based on `trend_pct < -10`, not a measured post-refresh business outcome. Therefore, the model does not establish that refreshing a recommended page will cause future SEO improvement.

The queue is appropriate for research and prioritization support. It is not a production publishing, deletion, rewriting, or autonomous SEO system.

In [24]:
# ML-10 Section 2 — Intended-use checks

print("INTENDED USE CHECK")
print("=" * 60)

print("Use case:")
print("- Ranked content-prioritization decision support")

print("\nTarget:")
print("- target_refresh = (trend_pct < -10)")

print("\nValidation evidence:")
print("- Client-grouped 80/20 split")
print("- Test clients were unseen during training")
print("- Week-4 baseline Precision@50: 52.0%")
print("- Random Forest Precision@50: 80.0%")
print("- Measured difference: +28.0 percentage points")

print("\nLimits:")
print("- Proxy target, not a post-refresh outcome")
print("- Directional evidence, not causal evidence")
print("- Human review required")
print("- Not a production automation system")

assert "trend_pct" not in features
assert client_column not in features
assert "content_id" not in features

print("\n✓ Target-defining field is excluded from features.")
print("✓ Identifiers are excluded from model features.")

INTENDED USE CHECK
Use case:
- Ranked content-prioritization decision support

Target:
- target_refresh = (trend_pct < -10)

Validation evidence:
- Client-grouped 80/20 split
- Test clients were unseen during training
- Week-4 baseline Precision@50: 52.0%
- Random Forest Precision@50: 80.0%
- Measured difference: +28.0 percentage points

Limits:
- Proxy target, not a post-refresh outcome
- Directional evidence, not causal evidence
- Human review required
- Not a production automation system

✓ Target-defining field is excluded from features.
✓ Identifiers are excluded from model features.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## Human review + the no-go list

Every recommended item must be reviewed by a person before action.

The reviewer should check:

1. Whether the page is still relevant to the business and audience.
2. Whether the observed traffic, impressions, clicks, ranking, and engagement signals make sense.
3. Whether the page has already been updated or is part of an active editorial project.
4. Whether the content has strategic, legal, brand, or customer importance that is not represented in the dataset.
5. Whether a refresh is actually the appropriate action rather than improving metadata, internal linking, technical SEO, or simply monitoring the page.
6. Whether the model recommendation is consistent with current human knowledge of the page.

### No-go cases

The model must not automatically:

- publish rewritten content;
- delete or prune a page;
- merge pages;
- change URLs;
- change canonical tags;
- change important business or legal claims;
- override an editor or subject-matter expert;
- treat a high score as proof that refreshing a page will improve SEO;
- make irreversible content changes without human approval.

The model score is a prioritization signal, not an instruction.

In [25]:
# ML-10 Section 3 — Human-review and no-go checks

required_queue_columns = [
    "rank",
    "content_id",
    "model_probability",
    "action",
    "reason_code"
]

missing = [
    c for c in required_queue_columns
    if c not in action_queue.columns
]

print("HUMAN REVIEW CHECK")
print("=" * 60)

print("Required queue columns:", required_queue_columns)
print("Missing columns:", missing)

assert len(missing) == 0

print("\n✓ Queue contains ranking, model score, action, and reason code.")
print("✓ Human review is required before editorial action.")
print("✓ No automatic publishing/deletion/rewriting is specified.")

HUMAN REVIEW CHECK
Required queue columns: ['rank', 'content_id', 'model_probability', 'action', 'reason_code']
Missing columns: []

✓ Queue contains ranking, model score, action, and reason code.
✓ Human review is required before editorial action.
✓ No automatic publishing/deletion/rewriting is specified.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## Monitoring / retrain triggers

The playbook should be reviewed periodically rather than assumed to remain valid indefinitely.

The following signals should trigger investigation:

- Precision@50 falls materially below the previously measured 80.0% validation result when evaluated on a comparable labeled sample.
- The distribution of model scores changes substantially from the validation period.
- The share of items receiving each action changes sharply without an obvious data reason.
- Key input features such as impressions, clicks, ranking, engagement, or freshness become unavailable or change definition.
- The target definition changes.
- The content population or client mix changes materially.
- Editorial feedback shows repeated false positives or false negatives.
- A new labeled outcome becomes available that better represents the real business result.

A retrain should be considered when performance degrades on a comparable evaluation sample, the data-generating process changes, or the target definition changes.

A scheduled retrain should not be treated as automatically necessary simply because time has passed.

In [26]:
# ML-10 Section 4 — Monitoring snapshot

print("MONITORING SNAPSHOT")
print("=" * 60)

# Score distribution
score_summary = action_queue["model_probability"].describe()

print("\nModel score distribution:")
display(score_summary.to_frame("model_probability"))

# Action distribution
action_distribution = (
    action_queue["action"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename("percent")
    .reset_index()
)

action_distribution.columns = ["action", "percent"]

print("\nAction distribution:")
display(action_distribution)

# Feature availability
feature_missing = (
    queue_df[features]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .rename("missing_percent")
    .reset_index()
)

feature_missing.columns = ["feature", "missing_percent"]

print("\nFeature missingness:")
display(feature_missing)

print("\nRETRAIN / REVIEW TRIGGERS")
print("- Comparable Precision@50 falls materially below 80.0%.")
print("- Input feature definitions or availability change.")
print("- Target definition changes.")
print("- Client/content population changes materially.")
print("- Persistent false positives or false negatives are observed.")
print("- A better measured business outcome becomes available.")

MONITORING SNAPSHOT

Model score distribution:


,model_probability
count,30000.000000
mean,0.522560
std,0.183927
min,0.000100
25%,0.448175
50%,0.571700
75%,0.651125
max,0.832200



Action distribution:


,action,percent
0,monitor,63.32
1,protect_monitor,33.74
2,improve_review,2.03
3,refresh_review,0.91



Feature missingness:


,feature,missing_percent
0,impressions_90d,0.0
1,clicks_90d,0.0
2,ctr,0.0
3,avg_position,0.0
4,sessions_90d,0.0
5,pageviews_90d,0.0
6,users_90d,0.0
7,engaged_sessions_90d,0.0
8,engagement_rate,0.0
9,days_since_last_update,0.0



RETRAIN / REVIEW TRIGGERS
- Comparable Precision@50 falls materially below 80.0%.
- Input feature definitions or availability change.
- Target definition changes.
- Client/content population changes materially.
- Persistent false positives or false negatives are observed.
- A better measured business outcome becomes available.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Exports for the paper

The ranked action queue is exported to `work/outputs/`.

The queue contains pseudonymous identifiers only and does not contain client names, URLs, or private search queries.

A compact metrics JSON is also exported so that the paper can trace its main measured result back to a reproducible file.

The queue is intentionally kept out of Git according to the assignment's data-handling design. The notebook regenerates it when executed.

In [27]:
# ML-10 Section 5 — Export paper-ready files

import json
from pathlib import Path

OUTPUT_DIR = REPO / "work/outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 1. Export ranked queue
# --------------------------------------------------

QUEUE_PATH = OUTPUT_DIR / "content_action_queue.csv"

action_queue.to_csv(
    QUEUE_PATH,
    index=False
)

# --------------------------------------------------
# 2. Export action summary
# --------------------------------------------------

ACTION_SUMMARY_PATH = OUTPUT_DIR / "action_summary.csv"

action_summary = (
    action_queue["action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="count")
)

action_summary["percent"] = (
    action_summary["count"]
    / len(action_queue)
    * 100
).round(2)

action_summary.to_csv(
    ACTION_SUMMARY_PATH,
    index=False
)

# --------------------------------------------------
# 3. Export metrics receipt
# --------------------------------------------------

METRICS_PATH = OUTPUT_DIR / "ml10_metrics.json"

metrics = {
    "dataset_rows": int(len(df)),
    "target": "trend_pct < -10",
    "model": "Random Forest",
    "validation_design": "client-grouped 80/20 split",
    "baseline_precision_at_50": 0.52,
    "random_forest_precision_at_50": 0.80,
    "measured_improvement_percentage_points": 28.0,
    "interpretation": "directional decision-support evidence",
    "target_type": "rule-derived proxy",
    "queue_rows": int(len(action_queue)),
    "human_review_required": True,
    "automatic_editorial_action": False
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

# --------------------------------------------------
# 4. Verify exports
# --------------------------------------------------

print("EXPORT CHECK")
print("=" * 60)

for path in [
    QUEUE_PATH,
    ACTION_SUMMARY_PATH,
    METRICS_PATH
]:
    print(path)
    print("Exists:", path.exists())
    if path.exists():
        print("Size:", path.stat().st_size, "bytes")
        print()

assert QUEUE_PATH.exists()
assert ACTION_SUMMARY_PATH.exists()
assert METRICS_PATH.exists()

# --------------------------------------------------
# 5. Verify queue contents
# --------------------------------------------------

saved_queue = pd.read_csv(QUEUE_PATH)

print("Saved queue shape:", saved_queue.shape)
print("Saved queue columns:")
print(saved_queue.columns.tolist())

display(saved_queue.head(10))

print("\n✓ Ranked queue exported successfully.")
print("✓ Action summary exported successfully.")
print("✓ Metrics receipt exported successfully.")

EXPORT CHECK
/content/flyrank-ml-internship/work/outputs/content_action_queue.csv
Exists: True
Size: 3296959 bytes

/content/flyrank-ml-internship/work/outputs/action_summary.csv
Exists: True
Size: 117 bytes

/content/flyrank-ml-internship/work/outputs/ml10_metrics.json
Exists: True
Size: 460 bytes

Saved queue shape: (30000, 10)
Saved queue columns:
['rank', 'content_id', 'client_id', 'model_probability', 'action', 'reason_code', 'days_since_last_update', 'impressions_90d', 'visibility_score', 'staleness_score']


,rank,content_id,client_id,model_probability,action,reason_code,days_since_last_update,impressions_90d,visibility_score,staleness_score
0,1,content_6ff59596237d,client_6208ef0f77,0.8322,improve_review,high_risk_stale,104,506,0.4442,0.8432
1,2,content_45c6b7efaa2b,client_6208ef0f77,0.8322,improve_review,high_risk_stale,104,563,0.4600,0.8432
2,3,content_6da8cc871a23,client_6208ef0f77,0.8316,improve_review,high_risk_stale,104,668,0.4850,0.8432
3,4,content_4b254381379e,client_6208ef0f77,0.8311,improve_review,high_risk_stale,104,903,0.5344,0.8432
4,5,content_9b6df29f7889,client_3fdba35f04,0.8291,improve_review,high_risk_stale,104,1622,0.6256,0.8432
5,6,content_98843f898048,client_6208ef0f77,0.8269,improve_review,high_risk_stale,104,712,0.4954,0.8432
6,7,content_51a87e64469f,client_6208ef0f77,0.8252,improve_review,high_risk_stale,104,931,0.5387,0.8432
7,8,content_0ba896169af5,client_4e07408562,0.8251,refresh_review,high_risk_stale_high_visibility,104,4998,0.7949,0.8432
8,9,content_11ba829169ac,client_19581e27de,0.8249,refresh_review,high_risk_stale_high_visibility,104,6124,0.8217,0.8432
9,10,content_1052cb379f1d,client_6208ef0f77,0.8242,improve_review,high_risk_stale,104,404,0.4137,0.8432



✓ Ranked queue exported successfully.
✓ Action summary exported successfully.
✓ Metrics receipt exported successfully.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [28]:
# ML-10 Final Self-check

print("ML-10 SELF-CHECK")
print("=" * 60)

checks = {
    "Action queue exists": QUEUE_PATH.exists(),
    "Action summary exists": ACTION_SUMMARY_PATH.exists(),
    "Metrics JSON exists": METRICS_PATH.exists(),
    "Queue is ranked": action_queue["rank"].is_monotonic_increasing,
    "Reason codes present": action_queue["reason_code"].notna().all(),
    "Actions present": action_queue["action"].notna().all(),
    "Model scores present": action_queue["model_probability"].notna().all(),
    "No trend_pct in features": "trend_pct" not in features,
    "No client identifier in features": client_column not in features,
    "No content identifier in features": "content_id" not in features,
}

for check, result in checks.items():
    print(f"{'✓' if result else '✗'} {check}")

assert all(checks.values())

print("\nML-10 self-check passed.")
print("The notebook is ready for Run All.")

ML-10 SELF-CHECK
✓ Action queue exists
✓ Action summary exists
✓ Metrics JSON exists
✓ Queue is ranked
✓ Reason codes present
✓ Actions present
✓ Model scores present
✓ No trend_pct in features
✓ No client identifier in features
✓ No content identifier in features

ML-10 self-check passed.
The notebook is ready for Run All.
